<a href="https://colab.research.google.com/github/Harshavardhan655/RL/blob/main/RL_LAB_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Assignment 11: Policy Iteration in a Deterministic Grid World

import numpy as np

# -----------------------------------------
# 1. Define the MDP
# -----------------------------------------

states = ["S0", "S1", "G"]

actions = ["Up", "Down", "Left", "Right"]

# Grid positions
positions = {
    "S0": (0, 0),
    "Wall": (0, 1),
    "S1": (1, 0),
    "G": (1, 1)
}

# Rewards and parameters
gamma = 0.9
normal_reward = -1
wall_reward = -1
goal_reward = 100
theta = 0.001

# Initial state values
V = {
    "S0": 0.0,
    "S1": 0.0,
    "G": 0.0
}

# Initial policy
policy = {
    "S0": "Up",
    "S1": "Up"
}

# Action movements
moves = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1)
}

# -----------------------------------------
# 2. Define Transition Function
# -----------------------------------------

def transition(state, action):

    if state == "G":
        return "G", 0

    row, col = positions[state]

    dr, dc = moves[action]

    new_row = row + dr
    new_col = col + dc

    # Check grid boundaries
    if new_row < 0 or new_row > 1 or new_col < 0 or new_col > 1:
        return state, normal_reward

    # Check wall
    if (new_row, new_col) == positions["Wall"]:
        return state, wall_reward

    # Check goal
    if (new_row, new_col) == positions["G"]:
        return "G", goal_reward

    # Normal movement
    if (new_row, new_col) == positions["S0"]:
        return "S0", normal_reward

    if (new_row, new_col) == positions["S1"]:
        return "S1", normal_reward


# -----------------------------------------
# 3. Policy Evaluation
# -----------------------------------------

def policy_evaluation(policy, V):

    while True:

        delta = 0

        new_V = V.copy()

        for state in ["S0", "S1"]:

            action = policy[state]

            next_state, reward = transition(state, action)

            value = reward + gamma * V[next_state]

            delta = max(delta, abs(value - V[state]))

            new_V[state] = value

        V = new_V

        if delta < theta:
            break

    return V


# -----------------------------------------
# 4. Calculate Action Values
# -----------------------------------------

def calculate_action_values(V):

    Q = {}

    for state in ["S0", "S1"]:

        Q[state] = {}

        for action in actions:

            next_state, reward = transition(state, action)

            Q[state][action] = reward + gamma * V[next_state]

    return Q


# -----------------------------------------
# 5. Policy Improvement
# -----------------------------------------

def policy_improvement(policy, V):

    Q = calculate_action_values(V)

    new_policy = policy.copy()

    for state in ["S0", "S1"]:

        best_action = max(Q[state], key=Q[state].get)

        new_policy[state] = best_action

    return new_policy, Q


# -----------------------------------------
# 6. Policy Iteration
# -----------------------------------------

iteration = 0

while True:

    iteration += 1

    print("\nPolicy Iteration:", iteration)

    # Evaluate current policy
    V = policy_evaluation(policy, V)

    # Improve policy
    new_policy, Q = policy_improvement(policy, V)

    print("State Values:", V)
    print("Current Policy:", policy)

    # Check policy stability
    if new_policy == policy:

        print("\nPolicy is stable!")
        break

    policy = new_policy


# -----------------------------------------
# 7. Display Final Results
# -----------------------------------------

print("\n========== FINAL RESULTS ==========")

print("\nOptimal State Values:")

for state in states:
    print(state, ":", V[state])

print("\nOptimal Policy:")

for state in ["S0", "S1"]:
    print(state, ":", policy[state])

print("G : Terminal")


# -----------------------------------------
# 8. Final Action Value Table
# -----------------------------------------

Q = calculate_action_values(V)

print("\n========== ACTION VALUE TABLE ==========")

print(f"{'State':<10}", end="")

for action in actions:
    print(f"{action:<12}", end="")

print()

for state in ["S0", "S1"]:

    print(f"{state:<10}", end="")

    for action in actions:
        print(f"{Q[state][action]:<12.2f}", end="")

    print()


# -----------------------------------------
# 9. Bellman Optimality Verification
# -----------------------------------------

print("\n========== BELLMAN VERIFICATION ==========")

for state in ["S0", "S1"]:

    max_Q = max(Q[state].values())

    print(
        f"V*({state}) = {V[state]:.2f}, "
        f"max Q*({state},a) = {max_Q:.2f}"
    )



Policy Iteration: 1
State Values: {'S0': -9.991404955442832, 'S1': -9.991404955442832, 'G': 0.0}
Current Policy: {'S0': 'Up', 'S1': 'Up'}

Policy Iteration: 2
State Values: {'S0': -9.993038013908693, 'S1': 100.0, 'G': 0.0}
Current Policy: {'S0': 'Up', 'S1': 'Right'}

Policy Iteration: 3
State Values: {'S0': 89.0, 'S1': 100.0, 'G': 0.0}
Current Policy: {'S0': 'Down', 'S1': 'Right'}

Policy is stable!

========== FINAL RESULTS ==========

Optimal State Values:
S0 : 89.0
S1 : 100.0
G : 0.0

Optimal Policy:
S0 : Down
S1 : Right
G : Terminal

========== ACTION VALUE TABLE ==========
State     Up          Down        Left        Right       
S0        79.10       89.00       79.10       79.10       
S1        79.10       89.00       89.00       100.00      

========== BELLMAN VERIFICATION ==========
V*(S0) = 89.00, max Q*(S0,a) = 89.00
V*(S1) = 100.00, max Q*(S1,a) = 100.00
